# 3.3 Your turn: events, not autocorrelations

[03.2](03.2-statistics-of-time.ipynb) found a cycle nobody was told about — in sunspots.
Your chat is not sunspots. A chat rarely hides an unknown rhythm: the daily and weekly
cycles are boring and known, everything else is bursty, one-off, and human. Point an
autocorrelation at it and you will mostly find lag 7, already knowing it was there —
and then go hunting for something, anything, in the other lags. That is **finding a
problem for your solution**: the tool came first and the question got invented to fit it.

This notebook runs the other direction, and the order is the whole exercise:

1. **Name an event you already know happened.** A holiday, a birthday, exam week, a trip,
   someone joining or leaving the group, a move, covid, a fight.
2. **Write down what impact it *should* have left — before looking.** More messages or
   fewer? Different people? Different hours? Longer or shorter messages, more emoji,
   another language?
3. **Then build the one plot that would show it — or fail to.** The tools are all from
   this lesson: reindexed daily counts, a rolling average, a date line, and 03.1's move —
   subtract the baseline, read what is left.

A prediction written down before looking is the difference between checking a hypothesis
and rationalising a squiggle.

In [ ]:
import pandas as pd
from goad_toolkit.datatransforms import (
    CountValues,
    Filter,
    FlagDates,
    Pipeline,
    RollingAvg,
    SortValues,
    SubtractBaseline,
    TimeFeatures,
)
from goad_toolkit.visualizer import (
    HorizontalLine,
    LinePlot,
    PlotSettings,
    VerticalDate,
)

from wa_analyzer.data import load_own_chat

own = load_own_chat()

## 3.3.1 Step one: events you know about

Fill the dict below from memory, not from the data — that is the point. If nothing comes
to mind yet, the fallback picks your busiest day ever and asks you the question in
reverse: *you* tell *it* what happened there. (Careful with the reverse direction: a spike
you then explain is a story, not a tested prediction — use it to get started, not to
conclude.)

In [ ]:
my_events = {
    # >>> Your turn: events you KNOW about, from memory -- "label": "YYYY-MM-DD" <<<
    # "trip to Rome": "2023-07-14",
    # "Sam joined the group": "2022-11-02",
}

daily = own.set_index("timestamp").resample("D").size().rename("messages").reset_index()

if not my_events:
    busiest = daily.loc[daily.messages.idxmax(), "timestamp"]
    my_events = {"busiest day ever -- what happened here?": str(busiest.date())}

for label, day in my_events.items():
    print(f"{day}  {label}")

## 3.3.2 Step two: what would the impact look like?

Write the expectation down first, next to the event. Different impacts live in different
plots — pick the plot from the prediction, never the other way around:

| if the event changed... | you would expect | the plot that would show it |
|---|---|---|
| **how much** | a spike or a dip in messages per day | daily counts + rolling average + a date line |
| **when** | the day's hourly shape shifts | hour shares around the event minus the ordinary shape — 03.1's residual move |
| **who** | someone appears, disappears, or takes over | messages per author, before vs after |
| **how** | length, emoji, links, questions, another language | a `RegexFeature` per marker (lesson 1), compared before vs after |

Two honesty checks before you trust anything you find. **Sample size**: one birthday is
one day — a difference between one day and the baseline can always be noise, and saying
so is a finding. **Shares, not counts**, whenever the volumes differ: a loud week wins
every raw comparison for boring reasons; 03.1 normalised each day to its own shares for
exactly this reason.

## 3.3.3 Worked example: the volume plot

Daily counts, a 7-day rolling average (`RollingAvg`, the same step as the flights
showcase — and the same warning: a wider window erases shorter events), and one
`VerticalDate` per event. `resample("D")` already emits every calendar day, quiet days
as zero — the honest-gap lesson from 03.1, handled at the counting step instead of
patched in afterwards.

In [ ]:
smoothed = Pipeline().add(
    RollingAvg, column="messages", window=7, rename=True
).apply(daily)

volume = PlotSettings(
    figsize=(12, 4),
    title="Messages per day, with the events I remember marked",
    xlabel="",
    ylabel="messages per day",
    xtick_rotation=45,
)
lines = LinePlot(volume)
fig, ax = lines.plot(data=daily, x="timestamp", y="messages", color="#cccccc",
                     label="daily")
lines.plot_on(LinePlot(volume), data=smoothed, x="timestamp",
              y="messages_rolling_avg", color="crimson", label="7-day average")
for label, day in my_events.items():
    lines.plot_on(VerticalDate(volume), date=day, label=label)
ax.legend()

Read it against your prediction, not for surprises: did the event you named move the
line the way you said it would? A spike you predicted is evidence. A spike you noticed
and then explained is a hypothesis for the *next* check, not a conclusion of this one.

## 3.3.4 Worked example: did the *shape* of the day change?

03.1's move, on your own data. Flag a window around one event (`FlagDates` matches whole
calendar days, so a timestamp column works as-is), build the hourly share inside the
window and the ordinary hourly share outside it, and let `SubtractBaseline` leave the
difference. Zero line means "this hour looked like any other day"; everything else is the
event's own signature.

In [ ]:
label, day = next(iter(my_events.items()))
event_day = pd.Timestamp(day)
window = pd.date_range(event_day - pd.Timedelta(days=3),
                       event_day + pd.Timedelta(days=3))

flagged = (
    Pipeline()
    .add(TimeFeatures, column="timestamp", features=["hour"])
    .add(FlagDates, column="timestamp", dates=window, feature="in_window")
    .apply(own)
)

baseline = (
    Pipeline()
    .add(Filter, expr="not in_window")
    .add(CountValues, column="hour", feature="share", normalize=True)
    .add(SortValues, column="hour")
    .apply(flagged)
)

event_shape = (
    Pipeline()
    .add(Filter, expr="in_window")
    .add(CountValues, column="hour", feature="share", normalize=True)
    .add(SortValues, column="hour")
    .add(SubtractBaseline, column="share", baseline=baseline, on="hour",
         feature="difference")
    .apply(flagged)
)

shape = PlotSettings(
    figsize=(9, 4),
    title=f"The week around '{label}', minus an ordinary day",
    xlabel="hour",
    ylabel="share of messages, event week − ordinary",
)
lines = LinePlot(shape)
fig, ax = lines.plot(data=event_shape, x="hour", y="difference", marker="o",
                     color="steelblue")
lines.plot_on(HorizontalLine(shape), y=0)

## 3.3.5 Your turn, for real

Pick the event you care most about and the impact row from the table that matches your
prediction, then build that plot. The pieces you already have:

- **`FlagDates`** — any set of days becomes a boolean column to `Filter` or group on.
- **`CountValues(normalize=True)` / `Share(by=...)`** — shapes instead of volumes.
- **`SubtractBaseline`** — the residual move, whenever "compared to normal" is the claim.
- **`RegexFeature`** (lesson 1) — emoji, links, a word, a language marker, someone's
  catchphrase, as a column you can then compare before/after.
- **`FacetPlot`** — the same plot per author or per period, when "who" is the question.

This is also exactly what the `goad` MCP is for: ask your assistant to run
`goad_analysis_checklist` with your event and your predicted impact as the question, and
expect to be interviewed — the checklist will not let you skip past "what would change my
mind?". `goad_critique_visual` is waiting once you have a chart.

## 3.3.6 What to write down

1. **The event and the predicted impact — written before you plotted.** Quote yourself.
2. **The plot that could have falsified it**, and whether it did.
3. **One expected impact you could *not* find**, with the sample it would have needed to
   show up. A null result you can size is worth more than one you cannot.